# Cancer Data Extraction with LLMs

The goal for this event is to extract information from clinical notes around cancer cases using large language models (LLMs). Specifically, you will be detecting cancer diagnoses and attempting to determine values for cancer registry variables from the **North American Association of Central Cancer Registries (NAACCR)**. At the end, you will create some tools for connecting to the ClinicalTrials.gov database to identify relevant clinical trials specific to a patient.

This exercise is designed to give participants experience with generative AI from two different angles:
1. Using chat agents to explore data and documentation, and have the agent use that contextual information to help develop analytical code
2. Creating code that submits queries to an LLM, including prompts, tools, and document retrieval

Essentially, you will be using AI to write code that uses AI. The code generated here will hopefully demystify popular AI tools and help you understand what is going on beneath the hood of AI agents.

The exercise is broken into sections, each with a series of tasks around which you should interact with the model/interface of your choice.
- You may write as much or as little code by hand as you would like
- These prompts are suggestions. Feel free to add or change tasks, or deviate completely if you think of something interesting you'd like to pursue


## 1. Best Practices for Working with a Coding Agent

A coding agent (e.g. Claude Code, Codex) is a powerful collaborator, but getting the most out of it requires a different mindset than writing code alone. Check out the `coding_agent_best_practices.md` for some tips and tricks to help you get the most out of your sessions.


## 2. Dataset and Documentation Exploration

The dataset contains synthetic clinical notes for patients with a mix of cancer and non-cancer diagnoses.

For test cases, refer to `config.yaml`:
- `TEST_CASE_IDS`: 10 cancer, 10 non-cancer
- `CANCER_TEST_CASE_IDS`: 10 cancer (same ids as above)

### 2a. Ask the chatbot to describe the files in the dataset directory.

In [ ]:
# Example prompt:
"""
Look at the files in the dataset/ directory and describe what each one contains.
What format are they in, how large are they, and what kind of information
do they hold? Also read config.yaml and explain the configuration.
"""

### 2b. Have the model to write some code to display descriptive visualizations of the file contents.

In [ ]:
# Example prompt:
"""
Write code to load the clinical notes from notes_by_case.json and the
ground truth from ground_truth.csv. Create visualizations showing:
1) the distributions of values in each ground truth column, 2) the number of
notes per case, and 3) the average note length by cancer status.
Use matplotlib or seaborn.
"""

### 2c. Ask the model to provide information about ground truth categories by searching the manuals and/or the web.

In [ ]:
# Example prompt:
"""
Look at the ground truth columns in ground_truth.csv. For each NAACCR
variable represented, search the NAACCR manuals in the data directory
and/or the web to explain what it measures, what the valid codes are,
and how it is typically determined from clinical records.
"""

## 3. Cancer Detection

The first task is straightforward: given a patient's clinical notes, determine whether the patient has cancer.

We'll start with a **single case** to develop the prompt, then **loop through all the test cases** and evaluate against ground truth.

(The functions developed in this section can be reused or iterated upon in future steps)

### 3a. Write a function to submit prompts to an LLM using the OpenAI Responses SDK.

In [ ]:
# Example prompt:
"""
Write a reusable function called `call_llm(prompt, system_prompt=None, model='gpt-4.1-mini')`
that sends a prompt to an LLM using the OpenAI Responses API and returns
the response text. Use the openai Python package. Read the API key from
an environment variable. Include error handling for API failures.
Test it with a simple prompt like 'What is NAACCR?'
"""

### 3b. Write a function that loads notes from a single case and submits them as part of a prompt to determine if that patient has cancer.
- Grab notes from the `notes_by_case.json` file.
- Test it by sending one set of notes and comparing the answer with the matching value from `ground_truth.csv`.

In [ ]:
# Example prompt:
"""
Write a function called `detect_cancer(case_id)` that loads the clinical
notes for a given case_id from notes_by_case.json, sends them to the LLM
with a prompt asking whether this patient has cancer, and returns True or
False. Use a system prompt like 'You are a clinical oncology expert.'
Test it on one cancer case and one non-cancer case from TEST_CASE_IDS,
and compare the results to ground_truth.csv.
"""

### 3c. Create a method for submitting a followup prompt.
If the patient has cancer, determine the affected tissue.

In [ ]:
# Example prompt:
"""
Write a function called `detect_cancer_and_tissue(case_id)` that first
calls detect_cancer(). If the patient has cancer, send a follow-up prompt
using the same clinical notes asking the LLM to identify the primary
affected tissue/organ. Return a dict with keys 'has_cancer' (bool) and
'tissue' (str or None). Test it on a known cancer case and verify the
tissue matches the ground truth.
"""

In [ ]:
# Example prompt:
"""
Write a Pydantic model called TissueResult with fields: has_cancer (bool)
and tissue (Optional[str]). Write a function detect_tissue(case_id) that
calls detect_cancer() first, and if the patient has cancer, submits a
follow-up prompt using the same notes to identify the affected tissue.
Return a TissueResult. Test it on a case you know has cancer.
"""

### 3c. Determine the cancer status and affected tissue (if applicable) for all the cases in `TEST_CASE_IDS`.

In [ ]:
# Example prompt:
"""
Loop through all case IDs in TEST_CASE_IDS from config.yaml and run
detect_cancer_and_tissue() on each one. Collect the results into a
pandas DataFrame with columns: case_id, predicted_cancer, predicted_tissue.
Display the DataFrame. Add a short delay between API calls to avoid
rate limiting.
"""

### 3d. Compare the responses to the ground truth values, and visualize some performance metrics (e.g. accuracy, precision, recall, etc.). If you need more information, the chatbot should be able to explain the options and make suggestions.

In [ ]:
# Example prompt:
"""
Merge the predictions DataFrame with ground_truth.csv on case_id.
Calculate accuracy, precision, recall, and F1 score for the cancer
detection task using sklearn.metrics. Create a confusion matrix heatmap
and a summary bar chart of the metrics. Print a classification report
and highlight any misclassified cases.
"""

## 4. NAACCR Variable Coding

- Tool calling
- Information retrieval
- ReAct loops
- Structured outputs


In [ ]:
# Using a data model to format the output will make things predictable to parse.
# Here's an example for a variable code
from typing import Literal
from pydantic import BaseModel, Field

class VariableCodingResult(BaseModel):
    """Generic structured output for any cancer registry coding task."""
    variable_name: str = Field(description="Name of the variable")
    variable_id: str = Field(description="Variable ID")
    value: str = Field(description="Value assigned based on variable coding")
    reasoning: str = Field(description="A brief explanation of the assessment")
    confidence: Literal["low", "medium", "high"] = Field(description="Confidence level in the assigned code")

### 4a. Develop a method for determining the NAACCR primary site code for a case.
Define custom tools (e.g. look things up in the coding rules or manuals) and/or call built-in ones (e.g. web search) to retrieve information to add to the model context. Have the model structure the final output using a data model (such as the Pydantic model above) for consistent output parsing.

In [ ]:
# Example prompt:
"""
Write a function called `code_primary_site(case_id)` that determines the
NAACCR primary site code (Item #400) for a cancer case. Define tool functions
the LLM can call — e.g., `lookup_site_coding_rules(tissue)` to search the
NAACCR manuals, and optionally a web search tool. Use a ReAct-style loop
where the model can call tools, receive results, and reason before producing
a final answer. Return a VariableCodingResult with the ICD-O-3 topography
code. Test it on a known case.
"""

### 4b. Test your method's performance against the ground truth.

In [ ]:
# Example prompt:
"""
Run code_primary_site() on all cases in CANCER_TEST_CASE_IDS. Compare the
predicted primary site codes to the ground truth values. Calculate exact
match accuracy and display a table showing each case's predicted vs actual
code, the reasoning, and confidence level. Highlight mismatches.
"""

### 4c. Are there any ways you can iterate on the method to improve the performance?
For example, bring in other sources, include images (see the site-specific [breast guidelines](https://seer.cancer.gov/manuals/2025/AppendixC/Coding_Guidelines_Breast_2025.pdf) for an example image), etc.

In [ ]:
# Example prompt:
"""
Look at the cases where primary site coding was incorrect. What went wrong?
Try improving the approach — for example: add the SEER site-specific coding
guidelines as a tool the model can retrieve, include anatomical reference
images in the prompt (e.g. the breast guidelines PDF), expand the system
prompt with more detailed instructions, or try few-shot examples. Re-run
and compare the new accuracy to the original.
"""

### 4d. Try some of the other variables with ground truth values.
Note: many NAACCR variables have site-specific instructions which requires identifying the tissue and/or primary site first. We have included some site specific coding rules around histology, but others are available on the [NCI SEER website](https://seer.cancer.gov/manuals/2026/appendixc.html).

In [ ]:
# Example prompt:
"""
Adapt the code_primary_site() function into a more general
`code_naaccr_variable(case_id, variable_name, variable_id)` function that
can code any NAACCR variable. It should look up the site first, then retrieve
the relevant site-specific coding rules for that variable. Test it on
histology (Item #522) for a few cancer cases. Check the SEER manuals
in the data directory for site-specific histology coding rules.
"""

## 5. Connect to ClinicalTrials.gov API

Now try to connect the cancer information you've extracted to real-world clinical trials. The [ClinicalTrials.gov API v2](https://clinicaltrials.gov/data-api/api) provides free, unauthenticated access to study data.

We'll build two tools:
1. **Simple search** — query by condition and/or search terms
2. **Advanced search** — add demographic and geographic filters (location within North Carolina)

API reference:
- [Human-readbale API documentation](https://clinicaltrials.gov/data-api/api)
- [Machine-readable API spec](https://clinicaltrials.gov/api/oas/v2)
- [Query construction guide](https://clinicaltrials.gov/find-studies/constructing-complex-search-queries)


### 5a. Ask the chatbot to take a look at the documentation for the ClinicalTrials.gov API.
Create some basic functions around sending API requests to the `/studies` endpoint to search clinical trials that are recruiting. Some hints:
- It may be helpful to create a data model to validate request data
- Focus on the `query` and `filter` parameters since those will be the most useful
- Test with queries that should yield fewer results (e.g. using "University of North Carolina at Chapel Hill" as the facility, specific condition) that can be checked using the "Try" button on the API site



In [ ]:
# Example prompt:
"""
Read the ClinicalTrials.gov API v2 documentation at
https://clinicaltrials.gov/data-api/api and the machine-readable spec
at https://clinicaltrials.gov/api/oas/v2. Write functions to:
1) `search_trials(condition, other_terms=None)` — query the /studies
   endpoint for recruiting studies matching a condition
2) Create a Pydantic model to validate request parameters (query.cond,
   query.term, filter.overallStatus)
Test by searching for trials at 'University of North Carolina at Chapel
Hill' for a specific cancer type and verify against the website.
"""

### 5b. Create functions to make advanced queries based on the query construction information.
Some things to try:
- Use the patient's information to set search limits
- Get the latitude/longitude of the cities in the case metadata and filter by distance
- Include common synonyms (e.g. colon, colorectal)

In [ ]:
# Example prompt:
"""
Build an advanced trial search function `search_trials_for_patient(case_id)`
that uses the patient's extracted cancer information to find relevant trials.
Add filters for: 1) patient age and sex from the case metadata,
2) geographic proximity — geocode the patient's city to lat/long and use
the API's distance filter, 3) condition synonyms (e.g. 'colon cancer'
and 'colorectal cancer'). Use the query construction guide at
https://clinicaltrials.gov/find-studies/constructing-complex-search-queries
for advanced syntax. Test on a cancer case and display matching trials.
"""